# VNA — a wideband readout-frequency sweep on real hardware

A **vector-network-analyser** scan of the readout path: step the readout carrier across the whole
first Nyquist zone, take **1000 shots** at each frequency, and plot the amplitude versus frequency. On
a fridge setup the dip is the readout resonator; on the `xm650-loopback` board it is the DAC→ADC
analog response.

This notebook runs against a **real ZCU216** over `RemoteDriver` (not the co-simulator — the
0.1–7.9 GHz span needs the board's 8 GS/s converters), so it is **not** executed in CI. Server
setup: [docs/software/board-server.md](../docs/software/board-server.md).

It builds the sweep **two ways** and compares how long each takes to collect the data:

1. a from-scratch on-core `@kernel` that returns **every** shot's raw IQ (`read_real` / `read_imag`,
   no on-core reduction) — one frequency's 1000 shots already fill the 16 KB core RAM, so the
   781-point sweep is **781 reruns**;
2. a second kernel that **accumulates each shot's power** `re²+im²` on-core — one word per frequency,
   phase-insensitive (correct for a general channel, not just a phase-calibrated qubit) — and
   **computes the swept frequency code on-core** — so ~3600 frequencies fit per buffer and this sweep
   is a single **rerun**.

Both retune the readout drive (ch 1) and demod carrier (ch 2) as a **matched pair** on-core (the
ADC-rate demod code is 4× the DAC-rate drive code) and fire the **same** `781 × 1000` shots — so the
comparison isolates the acquisition overhead the two data layouts pay.

## Connect to the board

In [ ]:
import time

import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt

from riscq import run as rq
from riscq.driver.remote import RemoteDriver, upload_bundle
from riscq.lang import Array, ParamTable, compile_kernel, kernel
from riscq.map import ADC_BATCH, LEAD, READOUT_LEAD, SocMap, SocParams
from riscq.pulses import Pulse, envelopes, units

BOARD = "192.168.191.99"                   # the ZCU216's LAN address (or the full PYRO: uri)

drv = RemoteDriver(BOARD, 9091)
print("server:", drv.board.info())
print("bundles on the board:", drv.board.bundles())

# first time only — push the build up and load it (~100 MB, a minute on GbE):
# upload_bundle(drv, "xm650-loopback",
#               xsa="../build/xm650-loopback/top.xsa",                   # write_hw_platform export
#               params_json="../software/configs/xm650-loopback.json")   # the SAME JSON the build used
# info = drv.board.load("xm650-loopback")                                # full RF bring-up; returns info()
# print(info)
# assert info["mts_result"] == 0, "multi-tile sync missed its target latencies"

m = SocMap(SocParams.from_json(drv.board.get_params()))   # always matches the loaded bitstream
fs = units.sample_rate(m.params)
print(f"connected to '{m.params.name}': {m.params.qubit_num} cores, "
      f"{fs / 1e9:.0f} GS/s DACs, first Nyquist zone 0 – {fs / 2e9:.1f} GHz")

## The kernel — capture every shot's IQ

The readout lives in two `ParamTable`s: a **readout drive** on channel 1 (`ro["meas"]`, the physical
measurement tone) and a **demod carrier** on channel 2 (`demod["sq"]`, whose `dur` is the integration
window). The kernel takes the DAC-rate readout code `freq` as a runtime parameter — the **seated**
register word (`units.freq_to_code`, spec 12), written to the carrier register raw and rewritten
between reruns with no reload — and programs the demod at `4 * freq`, the matched ADC-rate code (4× the
seated DAC word is the seated 4× code).

Every shot fires the drive and the demod together on a fixed `period` grid. `read_res()` **halts**
until that shot's integral settles, so `read_real()` / `read_imag()` latch *this* shot; we write the
raw I and Q straight into `out` through a cursor and move on. There is **no on-core averaging** — the
host gets all 1000 raw IQ pairs per frequency.

`now`, `play`, `set_freq`, `read_real`, … are firmware intrinsics resolved by name at compile time
(hence the `# noqa: F821`), not Python functions.

In [ ]:
@kernel
def vna_shots(ro: ParamTable, demod: ParamTable, out: Array, shots: int, period: int, freq: int):
    """One readout frequency, `shots` shots, every shot's raw IQ integral into `out` (2*shots words,
    no on-core averaging). The readout drive (ch 1) and demod carrier (ch 2) retune as a matched pair
    — the ADC-rate demod code is 4x the DAC-rate drive code — then fire together on a fixed grid."""
    init_pulse_params(ro.pulses)               # noqa: F821  load the readout-drive slot
    init_pulse_params(demod.pulses)            # noqa: F821  load the demod-carrier slot
    set_freq(ro, freq)                         # noqa: F821  DAC-rate readout drive; freq = seated register word (spec 12)
    set_freq(demod, 4 * freq)                  # noqa: F821  ADC-rate demod = 4x the seated DAC word == seated 4*code (int32 wrap = fold)
    t = now() + period                         # noqa: F821  first grid slot (retune lands >> LEAD ahead)
    k = 0
    for s in range(shots):
        play(ro, ro["meas"], t)                # noqa: F821  drive covers the demod window
        play(demod, demod["sq"], t)            # noqa: F821  firing the demod IS the readout
        wait_until(t + READOUT_LEAD)            # noqa: F821  let the stale level drop
        read_res()                             # noqa: F821  HALT until this shot's integral settles
        out[k] = read_real()                   # noqa: F821  this shot's I
        out[k + 1] = read_imag()               # noqa: F821  this shot's Q
        k = k + 2
        t = t + period                         # noqa: F821  next slot

## Build the readout tables and compile

The readout drive is a flat square covering the window; the demod is a flat square whose length is the
integration window. Both carriers are overwritten per rerun by `set_freq`, so the table frequency is
only a nominal placeholder. The grid `period` is sized so each shot's idle head lets the readout path
ring down: `idle + LEAD + window + READOUT_LEAD`, rounded up to a multiple of 8 so the demod LO lands
at the same phase every shot.

In [ ]:
SHOTS = 1000                     # shots per frequency
WIN = 1024                         # demod integration window (batches)
IDLE = 200                       # idle head per shot — readout-path ring-down (batches)
F_NOMINAL = 1e9                  # placeholder carrier for table construction (set_freq overrides it)

ro = ParamTable(1, F_NOMINAL, {"meas": Pulse(envelopes.square(WIN * 16), freq_hz=F_NOMINAL, amp=0.5)})
demod = ParamTable(2, 0.0, {"sq": Pulse(envelopes.square(WIN * 4), amp=1.0)})

period = -(-(IDLE + LEAD + WIN + READOUT_LEAD) // 8) * 8            # fixed grid, multiple of 8

prog = compile_kernel(vna_shots, m, tables=dict(ro=ro, demod=demod),
                      out=Array(2 * SHOTS), shots=SHOTS, period=period)

print(f"compiled: {SHOTS} shots/point on a {period}-batch grid "
      f"({units.ns(period, m.params) / 1e3:.2f} us/shot), "
      f"out buffer = {prog.var_size('out') // 4} words")

## The frequency grid — and why we need reruns

0.1 → 7.9 GHz in 10 MHz (0.01 GHz) steps is **781 points**. `units.freq_to_code` maps each to its
**seated** DAC register word — the signed 16-bit per-sample phase code sitting in `data[31:16]` (spec
12); above the 4 GHz Nyquist the code simply wraps (7.9 GHz → the same code as −0.1 GHz), which is
exactly how the converter aliases — a full-Nyquist scan by design.

The whole dataset is `781 × 1000 × 2` int32 words ≈ 6 MB. The core's on-chip RAM is only 16 KB
(4096 words), which holds at most ~1800 IQ pairs at once — so the sweep **cannot** be one run. We
acquire **one frequency's 1000 shots per rerun** (2000 words, a comfortable fit), read them back, and
let the next rerun reuse the buffer. `rq.setup` loads the image once; each `rq.rerun` only rewrites
the `freq` scalar and releases reset — O(1) driver ops, one RPC to the board.

In [ ]:
F_START, F_STOP, F_STEP = 0.1e9, 7.9e9, 1e7            # 0.1 -> 7.9 GHz, 10 MHz step
freqs = np.round(np.arange(F_START, F_STOP + F_STEP / 2, F_STEP))    # 781 points
codes = np.array([units.freq_to_code(float(f), m.params) for f in freqs])   # seated DAC register words (spec 12)
npts = len(freqs)

total_words = npts * SHOTS * 2
print(f"{npts} points, {F_STEP / 1e6:.0f} MHz apart, {F_START / 1e9:.1f} -> {F_STOP / 1e9:.1f} GHz")
print(f"full dataset  = {total_words:,} words ({total_words * 4 / 1e6:.0f} MB)")
print(f"core RAM      = {m.params.mem_depth} words ({m.params.mem_depth * 4 / 1024:.0f} KB) "
      f"-> {npts} reruns of {SHOTS * 2} words each")

## Acquire — one rerun per frequency

In [ ]:
QUBIT = 1

rq.setup(drv, m, {QUBIT: prog})                            # load image once; reset held, core 1 parked
TIMEOUT = SHOTS * period * 4 + 20_000_000              # poll budget: ~4 cycles/batch + boot slack

iq = np.empty((npts, SHOTS), dtype=np.complex128)      # every shot's IQ, kept on the host
t0 = time.time()
for i, freq in enumerate(codes):
    out = rq.rerun(drv, m, {QUBIT: prog}, params={QUBIT: {"freq": int(freq)}},
                   results=["out"], timeout=TIMEOUT)[QUBIT]["out"]   # one RPC; re-asserts reset on return
    z = out.reshape(SHOTS, 2)
    iq[i] = z[:, 0] + 1j * z[:, 1]
    if i % 500 == 0 or i == npts - 1:
        print(f"  {i + 1:>4}/{npts}  f = {freqs[i] / 1e9:5.3f} GHz   ({time.time() - t0:.0f}s)")

raw_secs = time.time() - t0                            # data-collection wall time (for the comparison)
print(f"done: collected {iq.size:,} IQ samples in {npts} reruns, {raw_secs:.0f}s")

## Plot the average shot amplitude vs frequency

For each frequency we have 1000 complex shots; the VNA amplitude is the mean of their magnitudes
`mean(|I + jQ|)`. The full per-shot array `iq` is still in hand if you want per-frequency scatter,
phase, or a coherent average instead.

In [ ]:
amp = np.abs(iq).mean(axis=1)                          # average shot amplitude per frequency

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(freqs / 1e9, amp, lw=0.7, color="#1f77b4")
ax.set_xlabel("readout frequency (GHz)")
ax.set_ylabel("mean shot amplitude  |IQ|")
ax.set_title(f"VNA sweep — {npts} points x {SHOTS} shots  ({m.params.name})")
ax.grid(alpha=0.3)

nyq = units.sample_rate(m.params) / 2e9
ax.axvline(nyq, color="gray", ls="--", lw=0.8)
ax.text(nyq, ax.get_ylim()[1], f" Nyquist {nyq:.1f} GHz", va="top", fontsize=8, color="gray")
fig.tight_layout()
plt.show()

## A second implementation — accumulate the shots on-core (power)

The first kernel returned **every** shot, so one frequency's 1000 shots (2000 words) already filled the
buffer — hence one rerun per point. A VNA only needs the **per-frequency amplitude**, so let the core
reduce the shots itself and return one number per frequency.

The right reduction is the **squared amplitude** `re·re + im·im`, summed over shots. A general channel
(a plain tone, not a phase-calibrated qubit readout) has an arbitrary demod phase — `re` and `im` take
either sign and rotate with frequency — so a *coherent* `re`/`im` sum would partly cancel. `|z|²` is
phase-insensitive; the host takes `sqrt(mean)` to recover the RMS amplitude. Each `re`/`im` is
pre-shifted by `sh` before squaring so the int32 power sum can't overflow (the decoder bounds
`|re| < 2¹⁷·WIN`, so `sh` is sized from that; the 1000-shot average recovers the few bits it drops).

The swept frequency is **computed on-core** too — no host code array. The host seeds each range's start
with `units.freq_to_code(f_start)` — the **seated** register word, the 16-bit DAC code in `data[31:16]`
(spec 12) — plus a fractional Q16 step `dcq`. On-core a Q16 accumulator `cq` adds `dcq` each point;
`c = (cq >> 16) << 16` masks off the sub-code fraction to give the seated DAC code, so the same `c`
drives the readout (`set_freq(ro, c)`) and, ×4, the matched demod (`set_freq(demod, 4 * c)` — the
demod must track the tone the DAC *actually* plays, `4·code`, not `4·f`, so the fraction is dropped
first). `cq` is a plain int32, so its wrap past 2³¹ *is* the Nyquist fold (build is `-fwrapv`), and
`c >> 16` tracks the per-point `_freq_code` within 1 LSB.

With one word per frequency, ~3600 points fit per buffer — so this whole 781-point sweep is a **single
rerun** (the range-splitting loop below generalises to finer sweeps).

In [ ]:
@kernel
def vna_accum(ro: ParamTable, demod: ParamTable, out: Array, npts: int, shots: int,
              period: int, sh: int, c0q: int, dcq: int):
    """Sweep `npts` readout frequencies (DAC codes computed on-core, seated code c = (cq >> 16) << 16)
    and accumulate the SQUARED AMPLITUDE re*re + im*im of each shot into `out` (1 word/point). A
    general channel's demod phase is arbitrary (re/im take either sign and rotate with frequency), so a
    coherent re/im sum would partly cancel — |z|**2 is phase-insensitive, and the host takes
    sqrt(out/shots) << sh to get the RMS amplitude. Each re/im is pre-shifted by `sh` first so re*re
    can't overflow int32 (the decoder bounds |re| < 2**17 * WIN; `sh` is sized from that and the
    1000-shot average recovers the few dropped bits). `out` is .bss, re-zeroed every rerun."""
    init_pulse_params(ro.pulses)               # noqa: F821
    init_pulse_params(demod.pulses)            # noqa: F821
    t = now() + period                         # noqa: F821  first grid slot
    cq = c0q
    for i in range(npts):
        c = (cq >> 16) << 16                   # noqa: F821  seated DAC code: drop the sub-code fraction, so 4*c can't leak
        set_freq(ro, c)                        # noqa: F821  DAC-rate readout drive
        set_freq(demod, 4 * c)                 # noqa: F821  ADC-rate demod = 4x the seated code (matched pair; Nyquist fold via int32 wrap)
        for s in range(shots):
            play(ro, ro["meas"], t)            # noqa: F821
            play(demod, demod["sq"], t)        # noqa: F821
            wait_until(t + READOUT_LEAD)        # noqa: F821  let the stale level drop
            read_res()                         # noqa: F821  HALT until this shot's integral settles
            re = read_real() >> sh             # noqa: F821  pre-shift so re*re can't overflow int32
            im = read_imag() >> sh             # noqa: F821
            out[i] += re * re + im * im        # noqa: F821  accumulate |z|**2 (phase-insensitive)
            t = t + period                     # noqa: F821
        cq = cq + dcq                          # noqa: F821  Q16 step -> next seated code

### Compile once, split into ranges

`npts`, `c0q`, `dcq` are all **runtime** parameters, so one compiled image serves every range; `out`
(the only array, now **one word per point**) is sized at `FREQS_PER_RUN`. The RAM ceiling is ~3600
points, so the 781-point sweep is **1 rerun** (a finer sweep past ~3600 points would split into
several). Each range passes its own `c0q` (its first frequency as a seated DAC code, `units.freq_to_code`)
and the shared fractional `dcq` step; the `>> 16` inside the kernel turns the Q16 accumulator into the
folded DAC code.

In [ ]:
# pre-square shift: the decoder bounds |read_real| < 2^15 * ADC_BATCH * WIN over the window, so this
# sizes sh to keep the int32 power sum from overflowing across `shots` shots (worst case re=im=iq_max)
sh = 10
FREQS_PER_RUN = 1500                            # out = npts words now (1/point); RAM caps ~3600 points

prog2 = compile_kernel(vna_accum, m, tables=dict(ro=ro, demod=demod),
                       out=Array(FREQS_PER_RUN), shots=SHOTS, period=period, sh=sh)

dcq = round(F_STEP * (1 << 32) / fs)                            # fractional Q16 step (per-point code advance << 16)

ranges = [slice(i, min(i + FREQS_PER_RUN, npts)) for i in range(0, npts, FREQS_PER_RUN)]
print(f"{len(ranges)} frequency range(s) of <= {FREQS_PER_RUN} points; pre-square shift sh = {sh}")

rq.setup(drv, m, {QUBIT: prog2})
amp_accum = np.empty(npts)
t0 = time.time()
for r in ranges:
    n = r.stop - r.start
    c0q = units.freq_to_code(float(freqs[r.start]), m.params)      # range start = seated DAC code (spec 12)
    out = rq.rerun(drv, m, {QUBIT: prog2}, params={QUBIT: {"npts": n, "c0q": c0q, "dcq": dcq}},
                   results=["out"], timeout=n * SHOTS * period * 4 + 20_000_000)[QUBIT]["out"][:n]
    amp_accum[r] = np.sqrt(out.astype(float) / SHOTS) * (1 << sh)  # RMS |z| per frequency
    print(f"  {freqs[r.start] / 1e9:5.3f}–{freqs[r.stop - 1] / 1e9:5.3f} GHz   ({time.time() - t0:.0f}s)")

accum_secs = time.time() - t0
print(f"done: {npts} points in {len(ranges)} reruns, {accum_secs:.0f}s")

### Compare the data-collection time

Same `781 × 1000` shots either way, so the *physics* time is identical — the power kernel wins purely
by doing far fewer reruns (781 → 1 here) and returning **one word per frequency** instead of
`2 × 1000`. Both curves are phase-insensitive amplitude: the per-shot kernel plots `mean(|z|)`, the
power kernel `sqrt(mean(|z|²))` (RMS). Each is normalized to its own peak below, so the shapes overlay
directly (the constant mean-vs-RMS scale factor divides out).

In [ ]:
amp_n = amp / amp.max()                     # normalize each curve to its own peak (largest point = 1)
amp_accum_n = amp_accum / amp_accum.max()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [3, 1]})
ax1.plot(freqs / 1e9, amp_n, lw=0.7, color="#1f77b4", label="per-shot  mean|IQ|  (kernel 1)")
ax1.plot(freqs / 1e9, amp_accum_n, lw=0.7, color="#d62728", label="on-core  RMS|IQ|  (kernel 2)")
ax1.set_xlabel("readout frequency (GHz)"); ax1.set_ylabel("normalized amplitude")
ax1.set_title("both implementations agree"); ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

ax2.bar(["per-shot", "power"], [raw_secs, accum_secs], color=["#1f77b4", "#d62728"])
ax2.set_ylabel("data-collection time (s)"); ax2.set_title(f"{raw_secs / accum_secs:.0f}× faster")
for i, s in enumerate([raw_secs, accum_secs]):
    ax2.text(i, s, f"{s:.0f}s", ha="center", va="bottom", fontsize=9)
fig.tight_layout(); plt.show()

print(f"per-shot : {npts:>4} reruns, {npts * SHOTS * 2 * 4 / 1e6:>5.0f} MB back, {raw_secs:>6.0f}s")
print(f"power    : {len(ranges):>4} reruns, {npts * 4 / 1e3:>5.1f} KB back, {accum_secs:>6.0f}s")
print(f"identical {npts * SHOTS:,} shots fired -> same physics; the power kernel returns 1 word/point "
      f"instead of {SHOTS * 2}, so the whole sweep is {len(ranges)} rerun(s)")

## Disconnect

In [ ]:
drv.close()
print("disconnected")